# Classifying Galaxies
In Lecture 13 a CNN predicted a number, a galaxy's axis ratio, by maximizing a Gaussian likelihood. Today we'll use the same CNN to classify galaxies into, smooth, edge-on disk, or spiral, using the Galaxy Zoo labels for the same SDSS images. 

In [2]:
import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

## fetch the data

The balanced sample of [Galaxy10 SDSS](https://astronn.readthedocs.io/en/latest/galaxy10sdss.html) images
from Lecture 13 (Leung & Bovy; labels from Galaxy Zoo 2): 2,000 galaxies of each class, binned to
34 × 34 pixels in *gri*. It is cached in `../lec13_cnn/data/`; the code rebuilds the cache from the
original 210 MB file only if it is missing.

In [3]:
TRAIN_CACHE = '../lec13_cnn/data/galaxy10_subset.npz'
GROUPS = {'smooth': [1, 2], 'edge-on disk': [4, 5, 6], 'spiral': [7, 8, 9]}


def galaxy10():
    import h5py, urllib.request
    if not os.path.exists('Galaxy10.h5'):
        urllib.request.urlretrieve('https://www.astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5', 'Galaxy10.h5')
    return h5py.File('Galaxy10.h5', 'r')


def bin2x2(imgs):
    return imgs[:, :68, :68].reshape(-1, 34, 2, 34, 2, 3).mean((2, 4)).round().astype(np.uint8)


def build_train_cache(per_class=2000, seed=2026):
    """The balanced Lecture 13 sample."""
    with galaxy10() as f:
        g10 = np.array(f['ans'])
        rng = np.random.default_rng(seed)
        idx = np.concatenate([np.sort(rng.choice(np.flatnonzero(np.isin(g10, c)), per_class, replace=False))
                              for c in GROUPS.values()])
        lab = np.repeat(np.arange(len(GROUPS)), per_class)
        order = np.argsort(idx)
        idx, lab = idx[order], lab[order]
        imgs = f['images'][idx]
    os.makedirs(os.path.dirname(TRAIN_CACHE), exist_ok=True)
    np.savez_compressed(TRAIN_CACHE, images=bin2x2(imgs), labels=lab.astype(np.uint8), galaxy10_class=g10[idx],
                        galaxy10_index=idx, class_names=np.array(list(GROUPS)))


if not os.path.exists(TRAIN_CACHE):
    build_train_cache()
train_data = np.load(TRAIN_CACHE)
names = list(train_data['class_names'])
counts = np.bincount(train_data['labels'])
print(f'{counts.sum()} galaxies:', ', '.join(f'{nm} {c}' for nm, c in zip(names, counts)))

6000 galaxies: smooth 2000, edge-on disk 2000, spiral 2000


The same 60/20/20 split and per-band standardization as in Lecture 13.

In [4]:
def to_tensor(images):
    return torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2) / 255.


X, Y = to_tensor(train_data['images']), torch.tensor(train_data['labels'].astype(int))
n = len(Y)
perm = np.random.default_rng(13).permutation(n)

i_tr, i_va, i_te = perm[:int(0.6 * n)], perm[int(0.6 * n):int(0.8 * n)], perm[int(0.8 * n):]
band_mean = X[i_tr].mean(dim=(0, 2, 3), keepdim=True)
band_std = X[i_tr].std(dim=(0, 2, 3), keepdim=True)
X = (X - band_mean) / band_std
print(f'train {len(i_tr)}, validation {len(i_va)}, test {len(i_te)} (all balanced)')

train 3600, validation 1200, test 1200 (all balanced)


## categorical likelihood

For a label $y \in \{0, 1, \dots, K-1\}$, the network outputs $K$ the logits $z_k$. 
Softmax turns them into probabilities that are positive and sum to one:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}.$$

The likelihood of the observed label is just the probability given to it. 
For $N$ galaxies, the negative log-likelihood is

$$-\log L = -\sum_{i=1}^N \log p_{y_i}(x_i; w),$$

i.e. the cross-entropy loss. 
pytorch implements it under `F.cross_entropy`. Let's check it against the categorical likelihood implemented by hand 

In [5]:
torch.manual_seed(0)
z = torch.randn(5, 3) # logits for 5 galaxies, 3 classes
y = torch.tensor([0, 2, 1, 1, 0]) # labels

p = torch.exp(z) / torch.exp(z).sum(dim=1, keepdim=True)        # softmax
nll_by_hand = -torch.log(p[torch.arange(5), y]).mean()
print(f'by hand: {nll_by_hand:.6f}   F.cross_entropy: {F.cross_entropy(z, y):.6f}')

# binary classes K = 2
z2 = torch.tensor([0.3, 1.7])
print(f'softmax, class 1: {torch.softmax(z2, 0)[1]:.6f}   sigmoid(z1 - z0): {torch.sigmoid(z2[1] - z2[0]):.6f}')
# softmax = sigmoid!

by hand: 1.341200   F.cross_entropy: 1.341200
softmax, class 1: 0.802184   sigmoid(z1 - z0): 0.802184


## Train the CNN as a classifier

Let's train the CNN from last class to classify the galaxy images. 
We want the CNN to output 3 logits (instead of a single number).  

Below is the CNN implementation similar to last class. Modify it to output 3 **logits**

In [6]:
class GalaxyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 34 -> 17
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 17 -> 8
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())                         # global average pool
        self.head = nn.Linear(64, 3) # dense layer 
        
    def forward(self, x):
        
        return self.head(self.features(x)).squeeze(1)

Now train the CNN! Here's a rough outline for the training. The snippet below will save the CNN with best validation loss (`cnn`) and the CNN at the end of training (`cnn_late`). Plot the loss curves.

In [7]:
def logits_of(model, x):
    # get logits --- model(x) --- in eval mode. 
    model.eval()
    with torch.no_grad():
        return model(x)


torch.manual_seed(0)
cnn = GalaxyCNN()
torch.manual_seed(1)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3) # adam optimizer 

history, best_loss, t0 = [], np.inf, time.time()
for epoch in range(150):
    cnn.train()
    order = torch.randperm(len(i_tr))
    for k in range(0, len(i_tr), 64):
        j = i_tr[order[k:k + 64].numpy()]

        loss = F.cross_entropy(cnn(X[j]),Y[j]) # add categorical loss evaluated over j 

        opt.zero_grad()
        loss.backward()
        opt.step()
        # backpropagate to get gradients 

        # take optimizer step

    # validation loss
    z_tr, z_va = logits_of(cnn, X[i_tr]), logits_of(cnn, X[i_va])
    history.append((F.cross_entropy(z_tr, Y[i_tr]).item(), F.cross_entropy(z_va, Y[i_va]).item(),
                    (z_va.argmax(1) == Y[i_va]).float().mean().item()))
    if history[-1][1] < best_loss:
        # save best validation loss
        best_loss, best_state, best_epoch = history[-1][1], copy.deepcopy(cnn.state_dict()), epoch
history = np.array(history)
cnn_late = copy.deepcopy(cnn) # weights after all 150 epochs
cnn.load_state_dict(best_state) # early-stopped weights
print(f'lowest validation loss at epoch {best_epoch + 1}')


lowest validation loss at epoch 105


## assess the classification
Calculate and plot a confusion matrix on the test data. Report the accuracy, precision, and recall for each class.

In [8]:
# evaluate the logits of the CNN on X[i_te]
z_te = logits_of(cnn, X[i_te])
softz = torch.softmax(z_te,dim=1)
softarr = softz.numpy()

lab_predict = np.argmax(softarr, axis=1)
yarr = Y[i_te].numpy()

conf_mat = np.zeros((3,3))
for i in range(3):
    for j in range(3):
        conf_mat[i,j] = np.sum( (lab_predict==j) & (yarr==i))

print('confusion matrix:')
print(conf_mat)
recall, precision, F1score = [], [], []
for i in range(3): 
    recnum = np.sum(conf_mat[i])
    recden = conf_mat[i,i]
    rec = recden/recnum
    recall = recall + [rec]

    presnum = np.sum(conf_mat[:,i])
    pres = recden/presnum
    precision = precision + [pres]

    f1 = 2/(1/rec + 1/pres)
    F1score = F1score + [f1]

accuracy = np.trace(conf_mat)/np.sum(conf_mat)
recall, precision, F1score = np.array(recall), np.array(precision), np.array(F1score)
print(f'recall: {recall}, precision: {precision}')
print(f'accuracy: {accuracy}, F1: {F1score}')
# apply softmax

confusion matrix:
[[391.   0.  13.]
 [  8. 363.  11.]
 [ 19.  15. 380.]]
recall: [0.96782178 0.95026178 0.9178744 ], precision: [0.9354067  0.96031746 0.94059406]
accuracy: 0.945, F1: [0.9513382  0.95526316 0.92909535]


Instead of picking the most probable class as a classification, suppose we want to a catalog of spirals for follow up. We'll classify a galaxy a sprial if $p_{\rm spiral} > t$ where $t$ is some threshold. Sweep $t$ and plot an ROC curve --- x: false positive rate, y: true positive rate

In [9]:
threshs = np.linspace(0.01,1-0.01,1000)
spiral_probs = softarr[:,2]

tpos_rates, fpos_rates = [], []
for i in range(len(threshs)):
    thr = threshs[i]
    th_pos = spiral_probs > thr
    tru_pos = np.sum( (th_pos == 1) & (true_spirals == 1))
    fals_neg = np.sum((th_pos == 0) & (true_spirals == 1 ))

    fal_pos = np.sum( (th_pos == 1) & (true_spirals == 0))
    tru_neg = np.sum((th_pos == 0) & (true_spirals == 0))
    
    true_rate = tru_pos/(tru_pos + fals_neg)
    false_rate = fal_pos/(fal_pos + tru_neg)
    
    tpos_rates = tpos_rates + [true_rate]
    fpos_rates = fpos_rates + [false_rate]

tpos_rates, fpos_rates = np.array(tpos_rates), np.array(fpos_rates)

plt.plot(fpos_rates,tpos_rates,lw=2)
plt.title('True positive rate vs false positive rate')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.show()



NameError: name 'true_spirals' is not defined